<a href="https://colab.research.google.com/github/yse-eds-cert/code-along-notebooks/blob/main/live-sessions/capstone_prompt_2_demo/full_version/1_data_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Before you start

**1. Save a copy in Drive.** File → Save a copy in Drive. Delete "Copy of" from the name.

**2. This notebook uses two runtimes.** Colab only connects to Google Drive while the runtime is Python, so we run **one cell in Python** to mount Drive, then switch to **R** for the rest:

1. **Runtime → Change runtime type → Python 3**, then do Step 0.
2. **Runtime → Change runtime type → R**, then run everything else.

You'll do this at the top of every notebook in your capstone project, because each notebook starts a fresh session.

# 1_data_clean: Reshape and Join the Raw Stream Data

**Capstone Prompt 2 live demo.** Toy research question: *Do streams in more developed watersheds run warmer in summer?*

**The rule for this script:** read from `raw_data/` and write to `clean_data/`. Never change the raw files.

| | File | What's in it |
|---|---|---|
| **Reads** | `raw_data/stream_temp_raw.csv` | Monthly mean water temperature (°C) at 7 monitoring stations |
| **Reads** | `raw_data/station_info_raw.csv` | One row per station: river, % impervious surface in the watershed, elevation, location |
| **Writes** | `clean_data/stream_temp_clean.csv` | One row per station per month, with the station info attached |

The two main cleaning steps:
1. **`pivot_longer()`** reshapes the temperature data from wide (one column per month) to long (one row per station-month).
2. **`left_join()`** attaches the station info to every temperature reading.

*The data are made up for this demo.*

---
## Step 0: Mount Google Drive (Python runtime)

"Mounting" connects this notebook to your Google Drive so the code can read and write files there. Colab only mounts Drive while the runtime is **Python**.

1. Make sure the runtime is Python: **Runtime → Change runtime type → Python 3**.
2. Run the cell below.
3. In the pop-up, **pick the same Google account you're using in Colab and tick _Select all_**, then click Continue.
4. When the cell prints `Mounted at /content/drive`, you're connected.

**Backup: the Mount Drive button.** If the cell gives an error (for example `credential propagation was unsuccessful`) even after you tick *Select all*, mount from the sidebar instead, while you're still in Python:
1. Click the **folder icon** (Files) in the left sidebar.
2. At the top of the Files panel, click the **Mount Drive** button (the folder with the Google Drive logo).
3. You're connected when a `drive` folder (with `MyDrive` inside) appears in the Files panel.

In [ ]:
# Run this cell with a PYTHON runtime. Switch to R afterwards.
from google.colab import drive
drive.mount('/content/drive')

---
## Step 1: Set the working directory and load packages

**Switch the runtime to R now:** Runtime → Change runtime type → R.

A short comment "header" at the top of a script tells a reader what the script does, what it reads, and what it writes.

In [ ]:
# =====================================================================
# 1_data_clean.ipynb
# Project: stream_temp_project
# Purpose: Reshape and combine the raw stream temperature and station
#          files into one clean, analysis-ready file.
# Reads:   raw_data/stream_temp_raw.csv
#          raw_data/station_info_raw.csv
# Writes:  clean_data/stream_temp_clean.csv
# =====================================================================

# Point R at the project folder so every path below can be short
setwd("/content/drive/MyDrive/stream_temp_project")
getwd()

In [ ]:
library(readr)  # read_csv(), write_csv()
library(dplyr)  # rename(), mutate(), filter(), left_join(), anti_join()
library(tidyr)  # pivot_longer()

---
## Step 2: Read the raw data and look at it

In [ ]:
# Read both raw files. This script only ever READS from raw_data/.
temps_raw    <- read_csv("raw_data/stream_temp_raw.csv")
stations_raw <- read_csv("raw_data/station_info_raw.csv")

In [ ]:
temps_raw

In [ ]:
stations_raw

### What do we notice?

- **The temperature data is wide.** Each month is its own column, so one row holds 12 measurements. Most tidyverse tools (grouping, plotting, joining) work best with **one observation per row**: one station in one month.
- **The key columns have different names.** The station ID is `station_id` in one file and `Station` in the other, and the station file capitalizes its column names.
- **The station lists don't match.** `ST07` has temperatures but no station info. `ST08` has station info but no temperatures.
- **There are missing values.** `ST03` has no reading for February and `ST05` has none for August.

---
## Step 3: Reshape the temperature data with `pivot_longer()`

`pivot_longer()` stacks several columns into two new columns:
- `names_to`: a new column that holds the **old column names** (`jan`, `feb`, ...)
- `values_to`: a new column that holds the **values** from those cells (the temperatures)

| station_id | jan | feb | ... |
|---|---|---|---|
| ST01 | 2.2 | 2.5 | ... |

becomes

| station_id | month | water_temp_c |
|---|---|---|
| ST01 | jan | 2.2 |
| ST01 | feb | 2.5 |
| ... | ... | ... |

In [ ]:
temps_long <- temps_raw %>%
  pivot_longer(
    cols      = jan:dec,         # the 12 month columns to stack
    names_to  = "month",         # old column names go into a column called "month"
    values_to = "water_temp_c"   # the temperatures go into a column called "water_temp_c"
  )

temps_long

In [ ]:
# Check: 7 stations x 12 months = 84 rows
cat("Rows before pivot:", nrow(temps_raw), "\n")
cat("Rows after pivot: ", nrow(temps_long), "\n")

---
## Step 4: Two small fixes to the long data

- **Month order.** `month` is text, so R would sort it alphabetically (apr, aug, dec, ...). We add a `month_num` column (1 to 12) so months sort in calendar order.
- **Missing readings.** A row with no temperature tells us nothing, so we drop it. We'll note this in the README.

In [ ]:
temps_long <- temps_long %>%
  # month.abb is built into R: "Jan" "Feb" ... "Dec".
  # match() finds each month's position in that list, giving 1 to 12.
  mutate(month_num = match(month, tolower(month.abb))) %>%
  # Drop the station-months where the sensor recorded nothing
  filter(!is.na(water_temp_c))

cat("Rows after dropping missing temperatures:", nrow(temps_long), "\n")

---
## Step 5: Make the station file's column names match

To join two tables, R needs a **key**: a column that appears in both tables and means the same thing in each. Here the key is the station ID, but it's called `station_id` in one file and `Station` in the other. We make all the station file's column names lowercase, then rename `station` to `station_id`.

In [ ]:
stations_clean <- stations_raw %>%
  rename_with(tolower) %>%          # Station -> station, River -> river, ...
  rename(station_id = station)      # new_name = old_name, to match temps_long

stations_clean

---
## Step 6: Combine the two tables with `left_join()`

`left_join(x, y, by = "key")` keeps **every row of `x`** (the left table) and adds `y`'s columns wherever the key matches. If a row in `x` has no match in `y`, its new columns are filled with `NA`.

- **x = `temps_long`**: our temperature readings. We want to keep all of them.
- **y = `stations_clean`**: adds `river`, `pct_impervious`, `elevation_m`, `latitude`, and `longitude` to each reading.

In [ ]:
stream_temp_clean <- temps_long %>%
  left_join(stations_clean, by = "station_id")

stream_temp_clean

### Always check a join

1. **Did the number of rows change?** Each station appears only once in `stations_clean`, so a left join shouldn't add or drop rows.
2. **Which rows of `x` found no match?** Their new columns are `NA`.
3. **Which rows of `y` were left out?** `anti_join(y, x)` lists the rows of `y` with no match in `x`.

In [ ]:
# 1. Row counts before and after the join should be the same
cat("Rows before join:", nrow(temps_long), "\n")
cat("Rows after join: ", nrow(stream_temp_clean), "\n")

In [ ]:
# 2. Temperature stations with no station info (river is NA after the join)
stream_temp_clean %>%
  filter(is.na(river)) %>%
  distinct(station_id)

In [ ]:
# 3. Stations in the station file with no temperature data (left out by left_join)
anti_join(stations_clean, temps_long, by = "station_id")

**What the checks show:**
- **ST07** has temperatures but no row in the station file, so its `river`, `pct_impervious`, and other station columns are `NA`. We keep these rows for now, note the gap in the README, and leave ST07 out of any analysis that needs `pct_impervious`.
- **ST08** (Eightmile River) has station info but no temperature data, so the left join leaves it out. That's what we want, because our unit of observation is a temperature reading.

Whatever you decide, **write it down in your README**. Cleaning decisions are part of your methods.

---
## Step 7: Put the columns in a sensible order

In [ ]:
stream_temp_clean <- stream_temp_clean %>%
  select(station_id, river, month, month_num, water_temp_c,
         pct_impervious, elevation_m, latitude, longitude) %>%
  arrange(station_id, month_num)   # sort by station, then by calendar month

glimpse(stream_temp_clean)

---
## Step 8: Write the clean data to `clean_data/`

In [ ]:
write_csv(stream_temp_clean, "clean_data/stream_temp_clean.csv")

# Check: the file should now be in clean_data/
list.files("clean_data")

---
## Done! Next steps for Capstone Prompt 2

1. **Rename and move this notebook** (if you haven't yet): `1_data_clean.ipynb`, in `stream_temp_project/scripts/`.
2. **Create a GitHub repository** called `stream_temp_project` on github.com, with a README.
3. **Save each notebook to GitHub:** File → Save a copy in GitHub → choose your repo → set the file path to `scripts/1_data_clean.ipynb`.
4. **Upload the data** to `raw_data/` and `clean_data/` folders in the repo, if you're allowed to share it.
5. **Write the README:** project description, research question, data sources, folder guide, run order, and cleaning decisions.
6. **Post on Canvas:** your repo link, plus a few sentences on which technique(s) you used and why.

```
stream_temp_project/
├── raw_data/
│   ├── stream_temp_raw.csv
│   └── station_info_raw.csv
├── clean_data/
│   └── stream_temp_clean.csv      ← written by this notebook
├── figures/
└── scripts/
    ├── 0_project_setup.ipynb
    └── 1_data_clean.ipynb
```